# Module 4 - Topic 2: Working with API Keys & Authentication
**Generative AI Fellowship — Beginner**

In this demo you will:
- Store an API key safely in a .env file
- Load it with python-dotenv
- Pass it in the Authorization header
- Handle authentication errors (401, 429)

> **Setup:** Create a `.env` file in this folder with your API key before running.


In [ ]:
# Cell 1 - Install
# !pip install requests python-dotenv

In [ ]:
# Cell 2 - Import
import requests
import os
from dotenv import load_dotenv

print("Libraries imported.")

In [ ]:
# Cell 3 - Load the API Key From .env
# load_dotenv() reads the .env file and puts the values into memory
# NEVER write your actual key in this notebook

load_dotenv()

# Get the key from environment variables
api_key = os.getenv("OPENAI_API_KEY")

# Print only the first 10 characters to confirm it loaded
# Never print the full key
if api_key:
    print("Key loaded successfully:", api_key[:10] + "...")
else:
    print("Key not found — check your .env file")

In [ ]:
# Cell 4 - Pass the Key in the Authorization Header
# The most common authentication pattern for modern REST APIs

# Build the headers dict with the Bearer token
headers = {
    "Authorization": f"Bearer {api_key}"
}

print("Header built:")
# Show the header structure without revealing the full key
print("Authorization: Bearer", api_key[:10] + "...")

In [ ]:
# Cell 5 - Make an Authenticated Request
# We call the OpenAI API — the same one we used in Module 2
# This time we build the request manually to see how authentication works

response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json={
        "model": "gpt-4o-mini",
        "messages": [
            {"role": "user", "content": "In one sentence, what is the capital of Nigeria?"}
        ],
        "max_tokens": 50
    },
    timeout=10
)

print("Status code:", response.status_code)

In [ ]:
# Cell 6 - Read the Response
# If status is 200, read the response

if response.status_code == 200:
    data = response.json()
    answer = data["choices"][0]["message"]["content"]
    print("Response:", answer)
else:
    print("Request failed with status:", response.status_code)

In [ ]:
# Cell 7 - Simulate a 401 Error (Wrong Key)
# What happens when the key is wrong?

wrong_headers = {
    "Authorization": "Bearer this-is-a-wrong-key"
}

response_wrong = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=wrong_headers,
    json={
        "model": "gpt-4o-mini",
        "messages": [{"role": "user", "content": "Hello"}]
    },
    timeout=10
)

print("Status code:", response_wrong.status_code)
# 401 — Unauthorized

In [ ]:
# Cell 8 - Handle Authentication Errors Properly
# Check the status code and show a clear message for each case

if response_wrong.status_code == 200:
    print("Success")
elif response_wrong.status_code == 401:
    print("Authentication failed — your API key is wrong or missing")
    print("Fix: check your .env file and make sure the key is correct")
elif response_wrong.status_code == 429:
    print("Rate limit exceeded — you have made too many requests")
    print("Fix: wait before retrying")
else:
    print("Unexpected error:", response_wrong.status_code)

In [ ]:
# Cell 9 - The Danger of Hardcoding Keys
# NEVER do this — shown here only as an example of what NOT to do

# WRONG — key is visible in the code file
# client = OpenAI(api_key="sk-proj-abc123...")   # DO NOT DO THIS

# RIGHT — key comes from environment variable
safe_key = os.getenv("OPENAI_API_KEY")

print("The safe pattern:")
print("1. Store key in .env file")
print("2. Add .env to .gitignore")
print("3. Load with load_dotenv()")
print("4. Read with os.getenv()")